# NIFTY Options IV Surface Reconstruction
### Algorithm: Cross-sectional polynomial smile fit + LightGBM residual correction
### Compliance: Strictly causal — no look-ahead bias.

## 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

# ---------- Hyperparameters ----------
DATASET_PATH    = 'dataset.csv'
SUBMISSION_PATH = 'submission.csv'
SEP = '||'

# LightGBM residual model
LGB_PARAMS = dict(
    objective         = 'regression',
    metric            = 'rmse',
    learning_rate     = 0.05,
    num_leaves        = 31,
    min_child_samples = 10,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    n_estimators      = 300,
    random_state      = 42,
    verbose           = -1,
)
TRAIN_MIN_ROWS = 100   # minimum residual samples before LGB is used

## 2. Load Data

In [2]:
df = pd.read_csv(DATASET_PATH)
iv_cols = [c for c in df.columns if c not in ('datetime', 'underlying_price')]
ce_cols = [c for c in iv_cols if c.endswith('CE')]
pe_cols = [c for c in iv_cols if c.endswith('PE')]
print(f'Loaded: {len(df)} rows, {len(iv_cols)} IV cols ({len(ce_cols)} CE, {len(pe_cols)} PE)')
print(f'Missing cells to fill: {df[iv_cols].isna().sum().sum()}')

Loaded: 975 rows, 28 IV cols (14 CE, 14 PE)
Missing cells to fill: 5460


## 3. Cross-sectional Smile Prediction

In [3]:
def cs_predict(row, col, group, is_ce):
    k0 = int(col[12:-2])
    obs = sorted(
        [(int(c[12:-2]), float(row[c])) for c in group
         if c != col and pd.notna(row[c]) and float(row[c]) > 0],
        key=lambda x: x[0]
    )
    if len(obs) < 2:
        return np.nan

    bl = sorted([(k, v) for k, v in obs if k < k0], key=lambda x: -x[0])
    ab = sorted([(k, v) for k, v in obs if k > k0], key=lambda x:  x[0])

    if bl and ab:
        pts = sorted(bl[:4] + ab[:4], key=lambda x: x[0])
        sk = np.array([p[0] for p in pts])
        sv = np.array([p[1] for p in pts])
        try:
            dists = np.abs(sk - k0).astype(float)
            dists[dists < 50] = 50
            weights = 1.0 / dists
            cf = np.polyfit(sk, sv, min(2, len(sk) - 1), w=weights)
            pred = float(np.polyval(cf, k0))
            local_min = min(sv[np.argmin(np.abs(sk - k0))], sv.min())
            local_max = max(sv[np.argmax(np.abs(sk - k0))], sv.max())
            if pred < local_min * 0.5 or pred > local_max * 2.0:
                lK, lIV = bl[0]; rK, rIV = ab[0]
                return lIV + (k0 - lK) / (rK - lK) * (rIV - lIV)
            return pred
        except Exception:
            lK, lIV = bl[0]; rK, rIV = ab[0]
            return lIV + (k0 - lK) / (rK - lK) * (rIV - lIV)

    side_all = sorted(bl if bl else ab, key=lambda x: abs(x[0] - k0))
    obs_ks = [s[0] for s in side_all]
    going_otm = (k0 > max(obs_ks)) if is_ce else (k0 < min(obs_ks))
    side = sorted(side_all[:3] if going_otm else side_all[:2], key=lambda x: x[0])
    sk = [p[0] for p in side]; sv = [p[1] for p in side]
    try:
        return float(np.polyval(np.polyfit(sk, sv, min(1, len(sk) - 1)), k0))
    except Exception:
        return side[0][1]

## 4. Pass 1 — Build Per-strike Residual Table (Causal)

In [4]:
print('Pass 1: computing cross-sectional residuals ...')
residuals = {col: {} for col in iv_cols}

for i in range(len(df)):
    row = df.iloc[i]
    for col in iv_cols:
        if pd.notna(row[col]):
            is_ce = col.endswith('CE')
            group = ce_cols if is_ce else pe_cols
            p = cs_predict(row, col, group, is_ce)
            if pd.notna(p) and p > 0.005:
                residuals[col][i] = float(row[col]) - p
    if (i + 1) % 250 == 0:
        print(f'  row {i+1}/{len(df)}')

print(f'  residuals stored: {sum(len(v) for v in residuals.values())}')

Pass 1: computing cross-sectional residuals ...
  row 250/975
  row 500/975
  row 750/975
  residuals stored: 21839


## 5. LightGBM Residual Model

In [5]:
# Features: [strike, is_ce, smile_pred, prev_iv, prev_residual, row_idx]
# Trained on Pass-1 residuals only — strictly causal, no look-ahead.

strike_cache = {col: int(col[12:-2]) for col in iv_cols}

def _make_feature(col, row_i, smile_pred):
    k0    = strike_cache[col]
    is_ce = 1 if col.endswith('CE') else 0
    sp    = smile_pred if pd.notna(smile_pred) else 0.15

    # Last observed residual for this strike strictly before row_i
    past = sorted([(idx, r) for idx, r in residuals[col].items() if idx < row_i])
    if past:
        prev_res = past[-1][1]
        prev_iv  = sp + prev_res
    else:
        prev_res = 0.0
        prev_iv  = sp

    return [float(k0), float(is_ce), float(sp),
            float(prev_iv), float(prev_res), float(row_i)]


# Build training set from Pass-1 residuals
print('Building LGB training set from residuals ...')
train_X, train_y = [], []
for col, res_dict in residuals.items():
    is_ce = col.endswith('CE')
    group = ce_cols if is_ce else pe_cols
    for row_i, res_val in res_dict.items():
        sp = cs_predict(df.iloc[row_i], col, group, is_ce)
        if pd.notna(sp):
            train_X.append(_make_feature(col, row_i, sp))
            train_y.append(res_val)

print(f'  LGB training samples: {len(train_y)}')

lgb_model = None
if len(train_y) >= TRAIN_MIN_ROWS:
    lgb_model = lgb.LGBMRegressor(**LGB_PARAMS)
    feature_cols = ['strike','is_ce','smile_pred','prev_iv','prev_residual','row_idx']
    train_X_df = pd.DataFrame(train_X, columns=feature_cols)
    lgb_model.fit(train_X_df, np.array(train_y))
    print('  LGB model trained.')
else:
    print('  Not enough samples — LGB correction will be 0.')


def get_correction(col, before_row, predicted_iv=None):
    """Return LGB-predicted residual correction; 0.0 if model unavailable."""
    if lgb_model is None:
        return 0.0
    sp   = predicted_iv if predicted_iv is not None else 0.15
    feat = pd.DataFrame([_make_feature(col, before_row, sp)],
                        columns=['strike','is_ce','smile_pred','prev_iv','prev_residual','row_idx'])
    return float(lgb_model.predict(feat)[0])

Building LGB training set from residuals ...
  LGB training samples: 21839
  LGB model trained.


## 6. Pass 2 — Fill Missing Values

In [6]:
print('Pass 2: filling missing cells ...')
filled = df.copy()
fill_count = 0

for i in range(len(df)):
    row = df.iloc[i]
    for col in iv_cols:
        if pd.isna(row[col]):
            is_ce = col.endswith('CE')
            group = ce_cols if is_ce else pe_cols
            p = cs_predict(row, col, group, is_ce)
            if pd.isna(p):
                past_vals = [df.iloc[j][col] for j in range(i) if pd.notna(df.iloc[j][col])]
                p = float(np.mean(past_vals)) if past_vals else 0.15
            corr = get_correction(col, i, predicted_iv=p)
            filled.at[i, col] = max(0.005, p + corr)
            fill_count += 1
    if (i + 1) % 250 == 0:
        print(f'  row {i+1}/{len(df)}')

print(f'  filled: {fill_count} cells, remaining NaN: {filled[iv_cols].isna().sum().sum()}')

Pass 2: filling missing cells ...
  row 250/975
  row 500/975
  row 750/975
  filled: 5460 cells, remaining NaN: 0


## 7. Validation Score

In [7]:
print('Validation: computing MSE via hold-out on observed cells ...')
val_actuals = []
val_preds   = []

for i in range(len(df)):
    row = df.iloc[i]
    for col in iv_cols:
        if pd.notna(row[col]):
            actual = float(row[col])
            is_ce  = col.endswith('CE')
            group  = ce_cols if is_ce else pe_cols

            p = cs_predict(row, col, group, is_ce)
            if pd.isna(p):
                past_vals = [df.iloc[j][col] for j in range(i) if pd.notna(df.iloc[j][col])]
                p = float(np.mean(past_vals)) if past_vals else 0.15

            corr = get_correction(col, i, predicted_iv=p)
            pred = max(0.005, p + corr)

            val_actuals.append(actual)
            val_preds.append(pred)

val_actuals = np.array(val_actuals)
val_preds   = np.array(val_preds)
mse  = float(np.mean((val_actuals - val_preds) ** 2))
rmse = float(np.sqrt(mse))
mae  = float(np.mean(np.abs(val_actuals - val_preds)))

print('\nValidation Results')
print('------------------')
print(f'Validation MSE  : {mse:.8f}')
print(f'Validation RMSE : {rmse:.6f}')
print(f'Validation MAE  : {mae:.8f}')
print(f'N samples       : {len(val_actuals):,}')

Validation: computing MSE via hold-out on observed cells ...

Validation Results
------------------
Validation MSE  : 0.00001574
Validation RMSE : 0.003967
Validation MAE  : 0.00124877
N samples       : 21,840


## 8. Create Submission

In [8]:
rows = []
for col in iv_cols:
    miss = df[col].isna()
    for idx in df.index[miss]:
        rows.append({
            'id':    f"{df.loc[idx, 'datetime']}{SEP}{col}",
            'value': float(filled.loc[idx, col])
        })

sub = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
sub.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved: {SUBMISSION_PATH}')
print(f'Submission rows: {len(sub):,}')
print('\nDone.')

Saved: submission.csv
Submission rows: 5,460

Done.
